# PCB Defect Image Synthesis & Quality Filter Pipeline
**Bài báo tham chiếu**: *"A Synthetic Defect Image Generation Method for Enhanced Industrial Defect Detection Based on Inpainting Diffusion and Context-aware Mask Generation"* (Journal of Intelligent Manufacturing, 2026)

Notebook này được thiết kế để huấn luyện và thực thi toàn bộ pipeline trực tiếp trên **Google Colab (GPU T4 / A100)**:
1. **Step 1**: Context-Aware Mask Generation (DINOv2 backbone + Top-5 Sampling)
2. **Step 2**: LoRA Inpainting Diffusion ($r=8, \alpha=16$, 6-token Textual Inversion, 9-channel UNet, Latent Overwriting)
3. **Step 3**: Confidence-based Quality Filtering bằng YOLO11s (100 epochs AdamW, Target-aware IoU matching, loại 25% thấp nhất)

In [ ]:
# ── 1. Kiểm tra cấu hình GPU ─────────────────────────────
!nvidia-smi

## Bước 1: Clone Mã Nguồn từ GitHub & Chuẩn bị Dataset
1. Chạy lệnh bên dưới để clone toàn bộ mã nguồn dự án từ GitHub vào Colab.
2. Đưa file dữ liệu `pcb-defect-dataset` vào thư mục `data/`.


In [ ]:
# ── 1. Chuẩn bị mã nguồn dự án ─────────────────────────────
import os
from pathlib import Path

if Path("pcb_pipeline").exists() and Path("run_pipeline.py").exists():
    print("Mã nguồn đã sẵn sàng tại:", os.getcwd())
elif Path("/content/diffusion_colab.zip").exists():
    print("Phát hiện file package 'diffusion_colab.zip'. Đang giải nén...")
    !unzip -q -o /content/diffusion_colab.zip -d /content/diffusion
    %cd /content/diffusion
elif Path("/content/diffusion/pcb_pipeline").exists():
    %cd /content/diffusion
else:
    print("Đang clone mã nguồn từ GitHub...")
    !git clone https://github.com/amebett1/diffusion.git
    if Path("/content/diffusion").exists():
        %cd /content/diffusion

# Kiểm tra xem mã nguồn đã ở đúng vị trí chưa
if not (Path("run_pipeline.py").exists() and Path("pcb_pipeline").exists()):
    raise RuntimeError(
        "LỖI: Chưa có mã nguồn dự án trong máy ảo Colab!\n"
        "Vui lòng kiểm tra lại kết nối GitHub hoặc chạy 'python package_for_colab.py' "
        "trên máy tính để tạo file 'diffusion_colab.zip' và kéo thả vào Colab."
    )

# ── 2. Đưa Dataset vào data/pcb-defect-dataset ─────────────
DATASET_LINK = ""  # <-- Dán link tải dataset của bạn vào đây (Google Drive hoặc URL trực tiếp)

if DATASET_LINK:
    !pip install -q gdown
    !gdown "{DATASET_LINK}" -O dataset.zip
    !mkdir -p data/pcb-defect-dataset
    !unzip -q dataset.zip -d data/
    print("Đã tải và giải nén dataset thành công!")
else:
    # Nếu bạn lưu file zip dataset trên Google Drive cá nhân, bạn có thể bỏ chú thích 3 dòng sau:
    # from google.colab import drive
    # drive.mount("/content/drive")
    # !unzip -q /content/drive/MyDrive/pcb-defect-dataset.zip -d data/
    pass

# Kiểm tra dữ liệu ảnh
raw_dir = Path("data/pcb-defect-dataset")
has_images = raw_dir.exists() and any(raw_dir.glob("**/*.jpg"))
if not has_images:
    raise RuntimeError(
        "DỪNG LẠI: Chưa có dữ liệu ảnh trong 'data/pcb-defect-dataset/'!\n"
        "Pipeline cần dữ liệu ảnh bo mạch DeepPCB để huấn luyện và sinh khuyết tật.\n"
        "Vui lòng thực hiện một trong các cách sau:\n"
        "1. Dán link tải dataset (Google Drive hoặc URL trực tiếp) vào biến DATASET_LINK ở trên.\n"
        "2. Hoặc bỏ chú thích các dòng drive.mount để giải nén từ Google Drive của bạn.\n"
        "3. Hoặc tải trực tiếp thư mục 'pcb-defect-dataset' lên thư mục 'data/' của Colab rồi chạy lại ô này."
    )
else:
    print("Dữ liệu PCB đã sẵn sàng trong data/pcb-defect-dataset/")


## Bước 2: Cài đặt các thư viện phụ thuộc


In [ ]:
%pip install -q diffusers transformers accelerate peft ultralytics opencv-python pyyaml gdown


## Bước 3: Tiền xử lý dữ liệu & Chuẩn hóa đường dẫn
Lệnh bên dưới sẽ tự động thực hiện:
1. Phân chia tập ID (10% Low-Data) và NPI theo chuẩn bài báo.
2. Tự động sinh **799 ảnh phôi sạch** (`data/pcb_normal`) bằng thuật toán Telea Inpainting.
3. Chuẩn hóa đường dẫn tương đối để chạy mượt mà trên môi trường Colab Linux.


In [ ]:
# Tiền xử lý dữ liệu và đồng bộ hóa đường dẫn môi trường:
!python pcb_pipeline/data_module.py
!python setup_env_paths.py


## Bước 4: Chạy Toàn Bộ Pipeline từ A -> Z (One-Click Full Pipeline)
Chạy toàn bộ 3 bước của bài báo bằng GPU với đầy đủ quá trình huấn luyện thực tế:
- **Step 1**: Sinh mặt nạ ngữ cảnh CAMG (Top-5 sampling)
- **Step 2a**: Huấn luyện LoRA ($r=8, \alpha=16$) + Textual Inversion 10 epochs
- **Step 2b**: Sinh ảnh lỗi với Stable Diffusion Inpainting + Latent Overwriting (50 steps)
- **Step 3**: Huấn luyện YOLO11s Judge 100 epochs với AdamW
- **Step 3 Filter**: Lọc độ tin cậy Target-aware IoU, loại 25% thấp nhất và xuất `augmented_dataset.yaml`

In [ ]:
!python run_pipeline.py full \
    --image_dir data/pcb_normal \
    --real_defect_dir data/pcb-defect-dataset \
    --metadata_path data/train_metadata.json \
    --data_yaml data/id_setup.yaml \
    --output_dir outputs/pcb_synthesis \
    --lora_epochs 10 \
    --lora_batch 32 \
    --inpaint_steps 50 \
    --guidance 7.5 \
    --yolo_size s \
    --yolo_epochs 100 \
    --reject_ratio 0.25 \
    --device cuda

## Bước 5: Chạy Step 4 — Downstream Benchmark (Tái lập Table 3 & 6 của Bài Báo)
Sau khi đã có tập dữ liệu tổng hợp đạt chuẩn từ Step 3 (`augmented_dataset.yaml`), ta tiến hành huấn luyện đối chứng:
- **Baseline**: 10% Low-Data thực tế ban đầu
- **Augmented**: 10% Real + 75% Accepted Synthetic Images
Đánh giá trên mô hình: **YOLO11s** (100 epochs, AdamW optimizer theo chuẩn bài báo trang 8).
*(Lưu ý: Mặc định chỉ chạy 1 mô hình YOLO11 để tiết kiệm thời gian GPU Colab ~45 phút. Nếu muốn chạy cả 4 mô hình, bạn chỉ cần thêm: --models yolo11 yolov8 yolov9 rtdetr)*


In [ ]:
# ── Chạy Benchmark đối chứng (Mặc định chạy 1 mô hình YOLO11 để tiết kiệm thời gian Colab) ───────
# Bạn có thể mở rộng danh sách nếu muốn: --models yolo11 yolov8 yolov9 rtdetr
!python run_pipeline.py step4 \
    --models yolo11 \
    --split id \
    --base_yaml data/id_setup.yaml \
    --aug_yaml outputs/pcb_synthesis/filtered/augmented_dataset.yaml \
    --epochs 100 \
    --batch_size 8 \
    --output_dir outputs/benchmark \
    --device cuda


In [ ]:
# ── Hiển thị Bảng Báo cáo So sánh Table 3 / Table 6 ───────
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path

rep_md = Path('outputs/benchmark/benchmark_report.md')
if rep_md.exists():
    display(Markdown(rep_md.read_text(encoding='utf-8')))

csv_p = Path('outputs/benchmark/benchmark_comparison.csv')
if csv_p.exists():
    df = pd.read_csv(csv_p)
    cols = ['Model', 'Setup', 'Precision Base (%)', 'Precision Aug (%)', 'Delta Precision (pp)', 'Recall Base (%)', 'Recall Aug (%)', 'Delta Recall (pp)', 'mAP@50 Base (%)', 'mAP@50 Aug (%)', 'Delta mAP@50 (pp)', '% Gain']
    display(df[[c for c in cols if c in df.columns]])


## Bước 6: Trực quan hóa kết quả (Báo cáo & Hiển thị ảnh mẫu)

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

report_path = Path('outputs/pcb_synthesis/pipeline_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print("=" * 50)
    print("TỔNG KẾT PIPELINE:")
    print(f"  Tổng mask đã tạo:      {report.get('total_masks')}")
    print(f"  Tổng ảnh đã sinh:      {report.get('total_generated')}")
    print(f"  Tổng ảnh đạt chuẩn:    {report.get('total_accepted')}")
    print(f"  Tỷ lệ loại bỏ:         {report.get('rejected_ratio') * 100:.0f}%")
    print(f"  Thời gian chạy:        {report.get('elapsed_minutes')} phút")
    print(f"  Augmented YAML:        {report.get('augmented_yaml')}")
    print("=" * 50)

# Hiển thị mẫu các ảnh khuyết tật nhân tạo accepted
acc_meta_path = Path('outputs/pcb_synthesis/filtered/accepted_metadata.json')
if acc_meta_path.exists():
    accepted_items = json.loads(acc_meta_path.read_text(encoding='utf-8'))
    samples = accepted_items[:min(4, len(accepted_items))]

    fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, item in zip(axes, samples):
        img_p = item.get('filtered_image_path') or item.get('synthetic_image_path')
        if Path(img_p).exists():
            ax.imshow(Image.open(img_p))
            ax.set_title(f"{item['defect_type']}\nConf: {item.get('confidence', 0):.3f}", fontsize=10)
        ax.axis('off')
    plt.suptitle("Sample Accepted Synthetic Defect Images", fontsize=14, fontweight='bold')
    plt.show()

## Bước 7: Sao lưu Toàn Bộ Kết quả & Trọng số về Google Drive
Lưu trữ vĩnh viễn toàn bộ ảnh tổng hợp, checkpoints LoRA, model weights YOLO11 và báo cáo benchmark về Google Drive của bạn:

In [ ]:
BACKUP_DIR = '/content/drive/MyDrive/PCB_Project_Outputs'
!mkdir -p {BACKUP_DIR}
!cp -r outputs/* {BACKUP_DIR}/
print(f"Đã sao lưu toàn bộ kết quả (synthesis + benchmark) về: {BACKUP_DIR}")
